# RQ3 -- Noise & sampling robustness

*How robust is parameter recovery to observation noise, sparse sampling, and initial conditions, and does that hold across a range of (β, γ)?*

Narrates `E08_noise.py`, `E09_sampling.py`, `E10_initial_conditions.py`, and `E11_uq.py`, and ends with the project's headline finding from `E13_crossover.py`.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve() / "scripts"))
from plot_style import apply_style, SOLVER_COLOR, COMPARTMENT_COLOR
apply_style()

RESULTS = Path("..") / "results"

def load(exp_id):
    return json.loads((RESULTS / f"{exp_id}.json").read_text())


## Noise degradation (E08)

In [ ]:
d8 = load("E08")
sigmas = [c["sigma_frac"] for c in d8["curve"]]
rmse = [c["rmse"][0] for c in d8["curve"]]
lo = [c["rmse_ci_lo"][0] for c in d8["curve"]]
hi = [c["rmse_ci_hi"][0] for c in d8["curve"]]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(sigmas, rmse, "o-")
ax.fill_between(sigmas, lo, hi, alpha=0.2)
ax.set_xlabel("σ"); ax.set_ylabel("RMSE(β̂)"); ax.set_title("Noise degradation with bootstrap CI")
plt.show()


## Sampling & window truncation (E09)

Identifiability degrades sharply once the observation window ends before the epidemic peak.

In [ ]:
d9 = load("E09")
import pandas as pd
df = pd.DataFrame(d9["cells"])
df["rmse_beta"] = df["rmse"].apply(lambda r: r[0])
pivot = df.pivot_table(index="window_frac", columns="dt", values="rmse_beta")
display(pivot)


## Uncertainty quantification, four ways (E11)

Asymptotic, bootstrap, Monte Carlo, and MCMC overlaid; MCMC's Gelman-Rubin R̂ confirms chain convergence.

In [ ]:
d11 = load("E11")
print("rho(beta,gamma) =", d11["asymptotic"]["rho"])
print("MCMC R-hat =", d11["mcmc"]["rhat"], " ESS(beta,gamma) =", d11["mcmc"]["ess_beta"], d11["mcmc"]["ess_gamma"])
fig, ax = plt.subplots(figsize=(5.5, 5))
ell = np.array(d11["asymptotic"]["ellipse"])
boot = np.array(d11["bootstrap"]["thetas"])
mcmc = np.array(d11["mcmc"]["pooled_samples"])
ax.plot(ell[:, 0], ell[:, 1], "-", label="asymptotic 95%")
ax.scatter(boot[:, 0], boot[:, 1], s=4, alpha=0.4, label="bootstrap")
ax.scatter(mcmc[:, 0], mcmc[:, 1], s=4, alpha=0.4, label="MCMC")
ax.scatter(*d11["theta_true"], color="black", marker="x", s=80, label="truth")
ax.set_xlabel("β"); ax.set_ylabel("γ"); ax.legend()
plt.show()


## The headline finding: crossover σ*

Solver bias (deterministic, set by h) vs noise-induced std (rises with σ). Their intersection is σ*(h).

In [ ]:
d13 = load("E13")
row = d13["rows"][0]
bias = abs(row["solver_bias"][0])
sigmas = [c["sigma_frac"] for c in row["sigma_curve"] if c["sigma_frac"] > 0]
stds = [c["std"][0] for c in row["sigma_curve"] if c["sigma_frac"] > 0]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(sigmas, stds, "o-", label="std across replicates")
ax.axhline(bias, ls="--", color="crimson", label="solver bias(h)")
sigma_star = row["crossover_sigma_star"].get("beta")
if sigma_star:
    ax.axvline(sigma_star, ls=":", color="green", label=f"σ*={sigma_star:.4f}")
ax.set_xlabel("σ"); ax.set_ylabel("|error| in β̂"); ax.legend()
ax.set_title(f"Crossover σ* (solver={d13['solver']}, h={row['h']})")
plt.show()
print(d13["conclusion"])
